# Benchmark Grover's Search Algorithm

Grover's algorithm is a foundational quantum algorithm, designed to solve the **unstructured search problem**: finding one or more "marked" items within a set of $N$ possibilities when no structure can be exploited. Classically, solving this problem requires $O(N)$ queries to an oracle that checks whether a candidate is a solution. Grover's algorithm achieves a quadratic speedup, requiring only $O(\sqrt{N})$ oracle queries.

***

### The model

We implements **Grover's search** in its minimal form, we perform one call to the Grover operator (no repeated iterations), where the oracle marks a single solution.

The circuit prepares a uniform superposition over $n$ qubits, applies the Grover operator once (oracle + diffuser), then measures. With one iteration, the marked states are amplified but not necessarily dominant; this is useful for understanding the algorithm or when the number of solutions is large relative to the search space.

We take a simple predicate function for marking the good states: 
$$
f(x)= (x==C),
$$
for $x$ a quantum number of $n$ qubits, $x=0,1,\dots 2^n-1$, and C being some integer.

### The Scoring function
The score is defined in terms of the success propabability of the marked states. For an initial state,
$$|\psi\rangle = \sum_a a_x |x\rangle,$$
if there are $M$ marked states, the probability to obtain one of the marked states is given by  
$$P_{\text{good}} = \sum_{x\in M} |a_x|^2~.$$
The score is defined as $S=P_{\text{good}} $, which obtains values between zero and one (the perfect score).

***
***


In [1]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

import numpy as np
from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

from classiq import *

In [2]:
# ============================================================
# Fresh start: reset all generated report/results files
# Uncomment to start a new benchmarking project from scratch
#
# from project_reset import reset_benchmark_project
# reset_benchmark_project()
# ============================================================

In [3]:
from examples.grover import GroverExample

***
***
## Benchmarking a 4-qubits Grover

Define a specific example on 4 qubits:

In [4]:
example = GroverExample(problem_size=4, c=5)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3DLNHILpEX1px2bhZCowhk3Ns9Q


### Set a `ResultCollector` with a file name fixed for the current example

In [5]:
FILENAME = example.default_results_filename

In [6]:
collector = ResultCollector(FILENAME, build_each_time=True)

In [7]:
# Uncomment to clear the data of a previous run
#
# await collector.reset_file()

### Choose on which backend to run 

We can print the list of backends:

In [8]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `ResultCollector`.)*

In [9]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [10]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [11]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

### Run Benchmark

In [12]:
print(
    "=" * 10
    + f"  {example.name}-{example.problem_size} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  grover-4 (2026-05-06 11:48:22.574486)   ==========
2026-05-06 11:48:28.009881: Submit grover-4 for Classiq - simulator
marked_prob_sum: 0.923
2026-05-06 11:48:30.594653: Completed grover-4 for Classiq - simulator --> Score {'score': '1.016', 'execution_time': '0.014', 'circuit_depth': 168, 'circuit_width': 5, 'two_qubit_gate_count': 78}
2026-05-06 11:48:32.089665: Submit grover-4 for Classiq - simulator_statevector
** Report updated: grover-4 for Classiq - simulator
marked_prob_sum: 0.9084472656249958
2026-05-06 11:48:34.035788: Completed grover-4 for Classiq - simulator_statevector --> Score {'score': '1.000', 'execution_time': '0.014', 'circuit_depth': 169, 'circuit_width': 5, 'two_qubit_gate_count': 82}
** Report updated: grover-4 for Classiq - simulator_statevector


In [13]:
await collector.print_status()

========== (2026-05-06 11:48:34.929066)   ==========
grover-4 | Classiq - simulator | COMPLETED | score=1.016 | time=0.014 min
grover-4 | Classiq - simulator_statevector | COMPLETED | score=1.000 | time=0.014 min
